# Customer Churn Analysis - Feature Engineering

This notebook focuses on feature engineering and data preparation for machine learning models.

## Contents
1. Data Loading and Cleaning
2. Feature Creation
3. Feature Encoding
4. Feature Scaling
5. Feature Selection
6. Save Processed Data

In [ ]:
# Import libraries
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from src.config import (
    RAW_DATA_PATH, PROCESSED_DATA_PATH,
    CATEGORICAL_FEATURES, NUMERICAL_FEATURES, TARGET,
    RANDOM_STATE, TEST_SIZE
)
from src.data_preprocessing import (
    load_data, clean_data, create_features,
    encode_features, prepare_data
)

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

print(f"✅ Imports successful!")
print(f"📁 Project root: {project_root}")

## 1. Data Loading and Cleaning

In [ ]:
# Load raw data
df_raw = load_data()
print(f"Raw data shape: {df_raw.shape}")
df_raw.head()

In [ ]:
# Clean data
df_clean = clean_data(df_raw)
print(f"\nCleaned data shape: {df_clean.shape}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")
df_clean.head()

In [ ]:
# Check data types after cleaning
print("\nData types after cleaning:")
print(df_clean.dtypes)

In [ ]:
# Verify Churn column is now string
print(f"\nChurn column type: {df_clean[TARGET].dtype}")
print(f"Churn unique values: {df_clean[TARGET].unique()}")
print(f"\nChurn distribution:")
print(df_clean[TARGET].value_counts())

## 2. Feature Creation

We'll create several new features based on EDA insights:

In [ ]:
# Create new features
df_feat = create_features(df_clean)
print(f"\nData shape after feature creation: {df_feat.shape}")
print(f"New features added: {df_feat.shape[1] - df_clean.shape[1]}")

In [ ]:
# Display new features
new_features = [col for col in df_feat.columns if col not in df_clean.columns]
print(f"\nNew features created: {new_features}")
df_feat[new_features + [TARGET]].head(10)

In [ ]:
# Analyze new features
print("\nNew Feature Statistics:")
df_feat[new_features].describe()

In [ ]:
# Visualize new features vs churn
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Total minutes vs churn
df_feat[df_feat[TARGET] == 'No']['Total minutes'].hist(bins=30, alpha=0.5, color='blue', ax=axes[0, 0], label='No Churn')
df_feat[df_feat[TARGET] == 'Yes']['Total minutes'].hist(bins=30, alpha=0.5, color='red', ax=axes[0, 0], label='Churn')
axes[0, 0].set_title('Total Minutes Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Total Minutes')
axes[0, 0].legend()

# Total calls vs churn
df_feat[df_feat[TARGET] == 'No']['Total calls'].hist(bins=30, alpha=0.5, color='blue', ax=axes[0, 1], label='No Churn')
df_feat[df_feat[TARGET] == 'Yes']['Total calls'].hist(bins=30, alpha=0.5, color='red', ax=axes[0, 1], label='Churn')
axes[0, 1].set_title('Total Calls Distribution', fontweight='bold')
axes[0, 1].set_xlabel('Total Calls')
axes[0, 1].legend()

# Avg charge per minute vs churn
sns.boxplot(data=df_feat, x=TARGET, y='Avg charge per minute', ax=axes[1, 0], palette=['blue', 'red'])
axes[1, 0].set_title('Avg Charge per Minute by Churn', fontweight='bold')

# High CS calls flag
pd.crosstab(df_feat['High CS calls'], df_feat[TARGET], normalize='index').plot(
    kind='bar', ax=axes[1, 1], color=['blue', 'red']
)
axes[1, 1].set_title('Churn Rate by High CS Calls Flag', fontweight='bold')
axes[1, 1].set_xlabel('High CS Calls (0=No, 1=Yes)')
axes[1, 1].set_ylabel('Proportion')
axes[1, 1].legend(title='Churn')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 3. Feature Encoding

Convert categorical variables to numerical using one-hot encoding:

In [ ]:
# Encode features
df_encoded = encode_features(df_feat, drop_first=True)
print(f"\nData shape after encoding: {df_encoded.shape}")
print(f"Number of features: {df_encoded.shape[1]}")

In [ ]:
# Display encoded columns
print("\nAll columns after encoding:")
print(df_encoded.columns.tolist())

In [ ]:
# Check for any remaining missing values
print(f"\nMissing values after encoding: {df_encoded.isnull().sum().sum()}")
if df_encoded.isnull().sum().sum() > 0:
    print("\nColumns with missing values:")
    print(df_encoded.isnull().sum()[df_encoded.isnull().sum() > 0])

## 4. Data Splitting and Feature Scaling

In [ ]:
# Separate features and target
X, y = prepare_data(df_encoded)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts(normalize=True))

In [ ]:
# Split data for analysis
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set size: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set size: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTraining set churn rate: {y_train.mean()*100:.2f}%")
print(f"Test set churn rate: {y_test.mean()*100:.2f}%")

In [ ]:
# Demonstrate scaling (will be done properly in model training)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nBefore Scaling - Training Set Statistics:")
print(pd.DataFrame(X_train).describe().loc[['mean', 'std']].iloc[:, :5].round(2))

print("\nAfter Scaling - Training Set Statistics:")
print(pd.DataFrame(X_train_scaled, columns=X.columns).describe().loc[['mean', 'std']].iloc[:, :5].round(2))

## 5. Feature Selection

Analyze feature importance and correlations:

In [ ]:
# Calculate correlation with target
feature_correlations = pd.DataFrame(X).corrwith(y).sort_values(ascending=False)

print("Top 15 features correlated with churn:")
print(feature_correlations.head(15))

print("\nBottom 15 features correlated with churn:")
print(feature_correlations.tail(15))

In [ ]:
# Visualize top correlations
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top positive correlations
feature_correlations.head(15).plot(kind='barh', ax=axes[0], color='coral')
axes[0].set_title('Top 15 Positive Correlations with Churn', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Correlation Coefficient')
axes[0].invert_yaxis()

# Top negative correlations
feature_correlations.tail(15).plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 15 Negative Correlations with Churn', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Correlation Coefficient')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Check for multicollinearity
correlation_matrix = pd.DataFrame(X).corr()

# Find highly correlated features (> 0.9)
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.9:
            high_corr_pairs.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print("Highly correlated feature pairs (|r| > 0.9):")
    for pair in high_corr_pairs[:10]:  # Show first 10
        print(f"  {pair[0]} <-> {pair[1]}: {pair[2]:.3f}")
    print(f"\nTotal highly correlated pairs: {len(high_corr_pairs)}")
else:
    print("No highly correlated features found (|r| > 0.9)")

## 6. Save Processed Data

In [ ]:
# Save processed data
df_encoded.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"\nProcessed data saved to: {PROCESSED_DATA_PATH}")
print(f"Shape: {df_encoded.shape}")

In [ ]:
# Summary of feature engineering
print("\n" + "="*80)
print("FEATURE ENGINEERING SUMMARY")
print("="*80)
print(f"Original features: {df_raw.shape[1]}")
print(f"After cleaning: {df_clean.shape[1]}")
print(f"After feature creation: {df_feat.shape[1]}")
print(f"After encoding: {df_encoded.shape[1]}")
print(f"\nFinal feature count for modeling: {X.shape[1]}")
print(f"Total samples: {X.shape[0]}")
print(f"\nClass balance:")
print(f"  No Churn: {(y==0).sum()} ({(y==0).mean()*100:.1f}%)")
print(f"  Churn: {(y==1).sum()} ({(y==1).mean()*100:.1f}%)")

print("\n" + "="*80)
print("Key Engineered Features:")
print("="*80)
for feat in new_features:
    print(f"  ✓ {feat}")

In [ ]:
print("\n" + "="*80)
print("FEATURE ENGINEERING COMPLETE!")
print("="*80)
print("\nNext Steps:")
print("1. Model Training with Multiple Algorithms")
print("2. Hyperparameter Tuning")
print("3. Model Evaluation and Comparison")
print("4. SHAP Explainability Analysis")
print("\nProceed to notebook 03_modeling.ipynb")